# 🏥 Detección de picos por provincia con Hadoop Streaming (Google Colab)

Este notebook instala y configura **Hadoop** en un entorno Colab, realiza la **ingesta a HDFS** del CSV de reportes de salud y ejecuta un **MapReduce** (clave = `Provincia|YYYY-MM-DD`, valor = `1`) para detectar **picos de reportes por día** en cada provincia.

**Flujo:**
1) Subida del CSV a Colab  
2) Instalación de Java y Hadoop  
3) Configuración de HDFS y arranque de NameNode/DataNode  
4) Ingesta del CSV en HDFS  
5) Map/Reduce con Hadoop Streaming  
6) Lectura y análisis de resultados  
7) (Opcional) Map por *Síntomas* si tu CSV trae una columna de texto `Síntomas`  

> **Nota:** Si tu archivo no se llama `Datos_Simulados_Reportes.csv`, cámbialo en las celdas donde se indica.

In [1]:
# ⬆️ 0) Subir el CSV
from google.colab import files  # Importa el módulo files de google.colab para manejar archivos en Colab.
up = files.upload()  # Abre un widget para subir archivos. El archivo subido se almacenará en la variable 'up'.

Saving Datos_Simulados_Reportes.csv to Datos_Simulados_Reportes.csv


In [4]:
%%bash
set -e  # Sale inmediatamente si un comando falla.
apt-get update -y  # Actualiza la lista de paquetes disponibles.
apt-get install -y openjdk-8-jdk-headless  # Instala el kit de desarrollo de Java 8 headless (sin interfaz gráfica).
wget -qO- https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz | tar -xz -C /usr/local/  # Descarga Hadoop 3.3.6, lo descomprime y lo extrae en /usr/local/.
mv /usr/local/hadoop-3.3.6 /usr/local/hadoop  # Renombra la carpeta de Hadoop a 'hadoop'.
echo '✅ Java y Hadoop instalados.'  # Imprime un mensaje de confirmación.

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
openjdk-8-jdk-headless is already the newest version (8u482-ga~us1-0ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
mv: cannot move '/usr/local/hadoop-3.3.6' to '/usr/local/hadoop/hadoop-3.3.6': Directory not empty


CalledProcessError: Command 'b"set -e  # Sale inmediatamente si un comando falla.\napt-get update -y  # Actualiza la lista de paquetes disponibles.\napt-get install -y openjdk-8-jdk-headless  # Instala el kit de desarrollo de Java 8 headless (sin interfaz gr\xc3\xa1fica).\nwget -qO- https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz | tar -xz -C /usr/local/  # Descarga Hadoop 3.3.6, lo descomprime y lo extrae en /usr/local/.\nmv /usr/local/hadoop-3.3.6 /usr/local/hadoop  # Renombra la carpeta de Hadoop a 'hadoop'.\necho '\xe2\x9c\x85 Java y Hadoop instalados.'  # Imprime un mensaje de confirmaci\xc3\xb3n.\n"' returned non-zero exit status 1.

In [ ]:
# 2) Variables de entorno
import os, glob  # Importa los módulos os (para interactuar con el sistema operativo) y glob (para encontrar archivos).
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"  # Establece la variable de entorno JAVA_HOME.
os.environ["HADOOP_HOME"] = "/usr/local/hadoop"  # Establece la variable de entorno HADOOP_HOME.
os.environ["PATH"] += os.pathsep + "/usr/local/hadoop/bin"  # Agrega el directorio bin de Hadoop a la variable de entorno PATH.
os.environ["PATH"] += os.pathsep + "/usr/local/hadoop/sbin"  # Agrega el directorio sbin de Hadoop a la variable de entorno PATH.

streaming_jars = glob.glob("/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar")  # Busca el archivo JAR de Hadoop Streaming.
assert streaming_jars, "No se encontró el jar de Hadoop Streaming."  # Verifica que el archivo JAR se haya encontrado.
streaming_jars[0]  # Muestra la ruta del archivo JAR encontrado.

In [ ]:
# 3) Configurar HDFS (core-site.xml y hdfs-site.xml)
core_site = """
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>/usr/local/hadoop_tmp</value>
  </property>
</configuration>
"""  # Define el contenido del archivo core-site.xml como una cadena multilinea.

hdfs_site = """
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
</configuration>
"""  # Define el contenido del archivo hdfs-site.xml como una cadena multilinea.

open("/usr/local/hadoop/etc/hadoop/core-site.xml","w").write(core_site)  # Crea y escribe el contenido en core-site.xml.
open("/usr/local/hadoop/etc/hadoop/hdfs-site.xml","w").write(hdfs_site)  # Crea y escribe el contenido en hdfs-site.xml.
print("✅ Archivos de configuración escritos.")  # Imprime un mensaje de confirmación.

In [ ]:
%%bash
# 4) Formatear NN (primera vez) y arrancar daemons HDFS
set -e  # Sale inmediatamente si un comando falla.
hdfs namenode -format -force  # Formatea el NameNode de HDFS (esto borra datos existentes). El argumento -force evita la confirmación interactiva.
$HADOOP_HOME/bin/hdfs --daemon start namenode  # Inicia el daemon del NameNode en segundo plano.
$HADOOP_HOME/bin/hdfs --daemon start datanode  # Inicia el daemon del DataNode en segundo plano.
sleep 2  # Espera 2 segundos para asegurar que los daemons se inicien.
hdfs dfs -mkdir -p /user/$USER  # Crea el directorio de usuario en HDFS si no existe.
echo '📂 Raíz HDFS:'  # Imprime un encabezado.
hdfs dfs -ls / || true  # Lista el contenido de la raíz de HDFS. '|| true' evita que el script falle si el directorio está vacío.

## Ingesta del CSV en HDFS

In [ ]:
%%bash
set -e  # Sale inmediatamente si un comando falla.
hdfs dfs -mkdir -p /data/salud  # Crea el directorio /data/salud en HDFS si no existe.
mv "Datos_Simulados_Reportes.csv" input_data.csv # Renombra el archivo local para evitar problemas con caracteres especiales.
hdfs dfs -put -f input_data.csv /data/salud/rep_last_month.csv  # Copia el archivo CSV local a HDFS, sobrescribiendo si ya existe (-f).
echo '✅ Ingesta completa. Contenido /data/salud:'  # Imprime un mensaje de confirmación.
hdfs dfs -ls /data/salud  # Lista el contenido del directorio /data/salud en HDFS.

## Mapper y Reducer (Provincia × Día)
El **mapper** emite `Provincia|YYYY-MM-DD` → `1`. El **reducer** suma por clave.

In [ ]:
%%writefile mapper_prov_dia.py
# -*- coding: utf-8 -*-
import sys, csv  # Importa los módulos sys (para interactuar con el intérprete) y csv (para trabajar con archivos CSV).
from datetime import datetime  # Importa la clase datetime del módulo datetime para manejar fechas.

def normalize_prov(s):
    # Normaliza el nombre de la provincia.
    if s is None:
        return ""
    s = str(s).strip().title()  # Convierte a cadena, elimina espacios, pone la primera letra en mayúscula.
    s = s.replace("Limon", "Limón").replace("San Jose", "San José")  # Corrige nombres comunes.
    return s

def parse_date(s):
    # Intenta parsear la fecha en varios formatos.
    s = (s or "").strip()  # Convierte a cadena, elimina espacios.
    for fmt in ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%m/%d/%Y %H:%M"): # Define formatos de fecha posibles.
        try:
            return datetime.strptime(s, fmt).date().isoformat()  # Intenta parsear la fecha y la formatea en ISO 8601.
        except Exception:
            pass  # Si falla, intenta con el siguiente formato.
    return None  # Retorna None si ningún formato coincide.

reader = csv.reader(sys.stdin)  # Crea un lector de CSV que lee de la entrada estándar (stdin).
header = next(reader, None)  # Lee la primera fila como encabezado.

# Autodetección de columnas
idx_prov = idx_fecha = None  # Inicializa los índices de columna.
if header:
    low = {c.strip().lower(): i for i, c in enumerate(header)}  # Crea un diccionario con nombres de columna en minúsculas como claves e índices como valores.
    idx_prov = low.get("provincia")  # Obtiene el índice de la columna 'provincia'.
    idx_fecha = low.get("fecha y hora", low.get("fecha", low.get("fecha_hora")))  # Obtiene el índice de la columna de fecha (con varios nombres posibles).

# Respaldo si no se detecta por nombre (ajusta según tu CSV)
if idx_prov is None or idx_fecha is None:  # Si no se encontraron las columnas por nombre.
    idx_prov, idx_fecha = 2, 0  # Usa índices fijos (ajustar si es necesario según el CSV).

for row in reader:  # Itera sobre cada fila del archivo CSV.
    try:
        prov = normalize_prov(row[idx_prov])  # Normaliza el nombre de la provincia.
        d = parse_date(row[idx_fecha])  # Parsea la fecha.
        if prov and d:  # Si la provincia y la fecha son válidas.
            print(f"{prov}|{d}\t1")  # Emite la clave (Provincia|Fecha) y el valor 1, separados por tabulador.
    except Exception:
        continue  # Ignora las filas con errores.

In [ ]:
%%writefile reducer_sum.py
# -*- coding: utf-8 -*-
import sys  # Importa el módulo sys para interactuar con el intérprete.

current_key = None  # Inicializa la clave actual a None.
running = 0  # Inicializa el contador acumulado a 0.

for line in sys.stdin:  # Itera sobre cada línea de la entrada estándar (salida del mapper).
    line = line.strip()  # Elimina espacios al principio y al final de la línea.
    if not line:
        continue  # Ignora líneas vacías.
    try:
        key, val = line.split("\t", 1)  # Divide la línea en clave y valor usando el tabulador como separador.
        val = int(val)  # Convierte el valor a entero.
    except Exception:
        continue  # Ignora las líneas con formato incorrecto.

    if current_key is None:  # Si es la primera clave que se procesa.
        current_key = key  # Establece la clave actual.
        running = 0  # Reinicia el contador.

    if key != current_key:  # Si la clave actual es diferente a la clave anterior.
        print(f"{current_key}\t{running}")  # Imprime la clave anterior y su suma total.
        current_key = key  # Actualiza la clave actual.
        running = 0  # Reinicia el contador.

    running += val  # Suma el valor actual al contador.

if current_key is not None:  # Después de procesar todas las líneas, si queda una clave pendiente.
    print(f"{current_key}\t{running}")  # Imprime la última clave y su suma total.

## Ejecutar Hadoop Streaming (leer de HDFS, escribir a HDFS)

In [ ]:
%%bash
set -e  # Sale inmediatamente si un comando falla.
chmod +x mapper_prov_dia.py reducer_sum.py  # Da permisos de ejecución a los scripts del mapper y reducer.
hdfs dfs -rm -r -f /salida/provincia_dia || true  # Elimina el directorio de salida en HDFS si existe. '|| true' evita el error si no existe.

hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar \
  -D mapreduce.job.reduces=1 \
  -files mapper_prov_dia.py,reducer_sum.py \
  -mapper "python3 mapper_prov_dia.py" \
  -reducer "python3 reducer_sum.py" \
  -input /data/salud/rep_last_month.csv \
  -output /salida/provincia_dia  # Ejecuta el job de Hadoop Streaming con los scripts mapper y reducer, leyendo de la entrada en HDFS y escribiendo la salida en HDFS.
  # -D mapreduce.job.reduces=1: Configura el número de reducers a 1.
  # -files: Especifica los archivos que se distribuirán a los nodos.
  # -mapper: Especifica el comando para ejecutar el mapper.
  # -reducer: Especifica el comando para ejecutar el reducer.
  # -input: Especifica la ruta del archivo de entrada en HDFS.
  # -output: Especifica la ruta del directorio de salida en HDFS.

echo "✅ Job completado. Archivos en /salida/provincia_dia:"  # Imprime un mensaje de confirmación.
hdfs dfs -ls /salida/provincia_dia  # Lista los archivos en el directorio de salida en HDFS.
echo "👀 Vista rápida:"  # Imprime un encabezado.
hdfs dfs -cat /salida/provincia_dia/part-* | head -n 20  # Muestra las primeras 20 líneas del archivo de salida en HDFS.

## Traer resultados a local y analizar

In [ ]:
%%bash
hdfs dfs -get -f /salida/provincia_dia/part-00000 provincia_dia.tsv  # Copia el archivo de salida del reducer de HDFS al sistema de archivos local, sobrescribiendo si existe (-f).
head -n 10 provincia_dia.tsv  # Muestra las primeras 10 líneas del archivo descargado.

In [ ]:
import pandas as pd  # Importa la librería pandas para análisis de datos.
df = pd.read_csv("provincia_dia.tsv", sep="\t", header=None, names=["key","total"])  # Carga el archivo TSV descargado en un DataFrame de pandas.
df['Provincia'] = df['key'].str.split('|').str[0]  # Divide la columna 'key' en 'Provincia' usando '|' como separador.
df['Dia'] = df['key'].str.split('|').str[1]  # Divide la columna 'key' en 'Dia' usando '|' como separador.

# Top-5 días por provincia (mayor número de reportes)
top5 = (df.sort_values(["Provincia","total"], ascending=[True, False]) # Ordena el DataFrame por Provincia (ascendente) y total (descendente).
          .groupby("Provincia").head(5))  # Agrupa por Provincia y selecciona las primeras 5 filas de cada grupo (los 5 días con más reportes).
top5.head(20)  # Muestra las primeras 20 filas del DataFrame top5.

In [ ]:
# Exportar CSV con Top-5 por provincia (opcional)
top5.to_csv("top5_picos_por_provincia.csv", index=False)  # Guarda el DataFrame top5 en un archivo CSV local sin incluir el índice.
from google.colab import files  # Importa el módulo files de google.colab.
files.download("top5_picos_por_provincia.csv")  # Descarga el archivo CSV a la máquina local del usuario.

In [ ]:
# Necesitamos leer el archivo CSV original para obtener la columna de género.
original_df = pd.read_csv("Datos_Simulados_Reportes.csv")

# Contar la cantidad de reportes por género
genero_counts = original_df['genero'].value_counts()

# Crear el gráfico circular
plt.figure(figsize=(8, 8))
genero_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=['skyblue', 'lightcoral'])
plt.title('Distribución de Reportes por Género')
plt.ylabel('')  # Eliminar el label del eje y para un gráfico circular
plt.show()

---
### 🔚 Cerrar daemons (opcional)

In [ ]:
%%bash
$HADOOP_HOME/bin/hdfs --daemon stop datanode || true  # Detiene el daemon del DataNode. '|| true' evita el error si el daemon no está corriendo.
$HADOOP_HOME/bin/hdfs --daemon stop namenode || true  # Detiene el daemon del NameNode. '|| true' evita el error si el daemon no está corriendo.
echo '🛑 HDFS detenido.'  # Imprime un mensaje de confirmación.